# 📊 TrOCR Evaluation Pipeline trên Kaggle

Notebook này được trích xuất từ pipeline OCR chính, chuyên dụng để chạy **inference TrOCR** trên tập test, đánh giá các metrics (CER, NED, Accuracy), vẽ biểu đồ và phân tích lỗi.

## 📦 1. Cài đặt thư viện

In [ ]:
!pip install -q transformers rapidfuzz matplotlib seaborn pandas tqdm nest_asyncio fugashi ipadic unidic-lite

## 🛠️ 2. Imports & Setup

In [ ]:
import os
import re
import unicodedata
import time

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as seaborn
from PIL import Image
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein
from transformers import AutoImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

import nest_asyncio
nest_asyncio.apply()

# Hiển thị tiếng Nhật trên matplotlib nếu cần
plt.rcParams['font.family'] = 'sans-serif'
# plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'Arial'] # Uncomment nếu có font tiếng Nhật

## ⚙️ 3. Configuration

In [ ]:
# Đường dẫn tới thư mục model TrOCR
TROCR_DIR = "/kaggle/input/models/thoandanh/trocr-for-rec/transformers/default/3/trocr-manga109-jpbert-lora-stage2-final-merged"

# === ĐƯỜNG DẪN DATASET ===
# Bạn cần upload dataset lên Kaggle và sửa lại 2 đường dẫn dưới đây cho đúng
DATASET_ROOT = "/kaggle/input/your-dataset-name" # Thay bằng thư mục chứa data của bạn
TEST_IMG_DIR = os.path.join(DATASET_ROOT, "test")
GT_FILE = os.path.join(DATASET_ROOT, "rec_gt_test.txt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config ready  |  device = {DEVICE}")

## 🧠 4. Load TrOCR Model

In [ ]:
print("Loading TrOCR...")
try:
    processor = AutoImageProcessor.from_pretrained(TROCR_DIR, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(TROCR_DIR, local_files_only=True)
    model = VisionEncoderDecoderModel.from_pretrained(TROCR_DIR, local_files_only=True)

    for cfg_obj in (model.config, model.generation_config):
        cfg_obj.decoder_start_token_id = tokenizer.cls_token_id
        cfg_obj.bos_token_id           = tokenizer.cls_token_id
        cfg_obj.pad_token_id           = tokenizer.pad_token_id
        cfg_obj.eos_token_id           = tokenizer.sep_token_id
    
    model.generation_config.max_new_tokens = 96
    model.eval().to(DEVICE)
    print(f"✅ TrOCR loaded successfully on {DEVICE}")
except Exception as err:
    print(f"❌ TrOCR load failed: {err}")

## 📄 5. Đọc file Label (rec_gt_test.txt)

In [ ]:
def _normalize(text: str) -> str:
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", text))

data = []
try:
    with open(GT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            # Thông thường PaddleOCR dùng tab để phân tách: filename\tlabel
            parts = line.split('\t')
            if len(parts) >= 2:
                img_path = parts[0]
                label = parts[1]
            else:
                # Nếu lỡ dùng khoảng trắng
                parts = line.split(' ', 1)
                img_path = parts[0]
                label = parts[1] if len(parts) > 1 else ""
                
            # Lấy tên file gốc (ví dụ img_path có thể là "test/test_000001.png")
            filename = os.path.basename(img_path)
            full_path = os.path.join(TEST_IMG_DIR, filename)
            
            data.append({
                "image_path": full_path, 
                "filename": filename,
                "ground_truth": _normalize(label)
            })
            
    df = pd.DataFrame(data)
    print(f"✅ Đã load {len(df)} samples từ tập test.")
    display(df.head())
except FileNotFoundError:
    print(f"❌ Không tìm thấy file {GT_FILE}. Vui lòng kiểm tra lại cấu hình DATASET_ROOT.")

## 📏 6. Hàm Đánh Giá Metrics

In [ ]:
def calculate_metrics(true_text: str, pred_text: str):
    """
    Tính toán Character Error Rate (CER), Normalized Edit Distance (NED) và Exact Match.
    """
    if len(true_text) == 0 and len(pred_text) == 0:
        return 0.0, 1.0, 1
    
    dist = Levenshtein.distance(true_text, pred_text)
    max_len = max(len(true_text), len(pred_text))
    
    cer = dist / len(true_text) if len(true_text) > 0 else 1.0
    ned = 1.0 - (dist / max_len) if max_len > 0 else 1.0
    exact_match = 1 if true_text == pred_text else 0
    
    return cer, ned, exact_match

def run_trocr_inference(img_path: str):
    image = Image.open(img_path).convert('RGB')
    pv = processor(image, return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        ids = model.generate(
            pv, 
            max_new_tokens=96, 
            num_beams=4,
            early_stopping=True, 
            no_repeat_ngram_size=3, 
            repetition_penalty=1.3,
        )
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    return _normalize(text)

## 🚀 7. Chạy Inference Trên Tập Test

In [ ]:
results = []
error_count = 0

if 'df' in locals() and len(df) > 0:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="TrOCR Inference"):
        img_path = row['image_path']
        true_text = row['ground_truth']
        
        if not os.path.exists(img_path):
            error_count += 1
            continue
            
        try:
            pred_text = run_trocr_inference(img_path)
            cer, ned, match = calculate_metrics(true_text, pred_text)
            
            results.append({
                "filename": row['filename'],
                "image_path": img_path,
                "ground_truth": true_text,
                "prediction": pred_text,
                "cer": cer,
                "ned": ned,
                "exact_match": match
            })
        except Exception as e:
            print(f"Error processing {row['filename']}: {e}")
            error_count += 1

    res_df = pd.DataFrame(results)
    
    # Lưu kết quả inference để sau này kiểm tra lại
    res_df.to_csv("/kaggle/working/trocr_evaluation_results.csv", index=False)
    print(f"\n✅ Hoàn thành. Đã lưu kết quả tại /kaggle/working/trocr_evaluation_results.csv")
    if error_count > 0:
        print(f"⚠️ Bỏ qua {error_count} ảnh do lỗi (file không tồn tại hoặc lỗi đọc).")
else:
    print("⚠️ Chưa có dữ liệu để chạy inference.")

## 📊 8. Báo Cáo Tổng Quan (Overall Metrics)

In [ ]:
if 'res_df' in locals() and len(res_df) > 0:
    avg_cer = res_df['cer'].mean()
    avg_ned = res_df['ned'].mean()
    accuracy = res_df['exact_match'].mean() * 100

    print("="*40)
    print("🏆 EVALUATION SUMMARY 🏆")
    print("="*40)
    print(f"Tổng số mẫu hợp lệ : {len(res_df)}")
    print(f"Average CER        : {avg_cer:.4f}")
    print(f"Average NED        : {avg_ned:.4f}")
    print(f"Matching Accuracy  : {accuracy:.2f} %")
    print("="*40)

## 📈 9. Trực Quan Hóa (Biểu đồ CER & NED)

In [ ]:
if 'res_df' in locals() and len(res_df) > 0:
    # Set seaborn style
    seaborn.set_theme(style="whitegrid")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Biểu đồ phân phối CER
    # Giới hạn CER ở mức 2.0 để biểu đồ không bị kéo giãn quá mức bởi các outlier (CER có thể > 1)
    plot_cer = res_df['cer'].clip(upper=2.0)
    seaborn.histplot(plot_cer, bins=30, kde=True, ax=axes[0], color='salmon', edgecolor="black")
    axes[0].set_title('Character Error Rate (CER) Distribution', fontsize=14)
    axes[0].set_xlabel('CER (Clipped at 2.0)', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    
    # Biểu đồ phân phối NED
    seaborn.histplot(res_df['ned'], bins=30, kde=True, ax=axes[1], color='skyblue', edgecolor="black")
    axes[1].set_title('Normalized Edit Distance (NED) Distribution', fontsize=14)
    axes[1].set_xlabel('NED (Closer to 1 is better)', fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/metrics_distribution.png', dpi=300)
    plt.show()

## 🕵️‍♂️ 10. Phân Tích Lỗi (Error Analysis)

Hiển thị những trường hợp model dự đoán sai nhiều nhất (có Normalized Edit Distance thấp nhất).

In [ ]:
if 'res_df' in locals() and len(res_df) > 0:
    TOP_K = 10
    # Sắp xếp tăng dần theo NED (càng thấp nghĩa là càng khác biệt)
    error_cases = res_df.sort_values(by='ned', ascending=True).head(TOP_K)
    
    print(f"🚨 TOP {TOP_K} SAMPLES CÓ SAI SỐ CAO NHẤT 🚨\n")
    
    for i, (_, row) in enumerate(error_cases.iterrows(), 1):
        print(f"{i}. {row['filename']}")
        print(f"   GT   : {row['ground_truth']}")
        print(f"   Pred : {row['prediction']}")
        print(f"   CER  : {row['cer']:.4f} | NED: {row['ned']:.4f}")
        print("-"*50)
        
    # (Tùy chọn) Hiển thị ảnh của top 5 lỗi lớn nhất
    num_display = min(5, len(error_cases))
    if num_display > 0:
        fig, axes = plt.subplots(num_display, 1, figsize=(8, 2 * num_display))
        if num_display == 1:
            axes = [axes]
            
        for i in range(num_display):
            row = error_cases.iloc[i]
            try:
                img = Image.open(row['image_path'])
                axes[i].imshow(img)
                axes[i].axis('off')
                
                # Vì có thể không hiển thị được tiếng Nhật nếu thiếu font, ta in text ra console ở trên rồi,
                # trên biểu đồ chỉ ghi số liệu NED.
                axes[i].set_title(f"{row['filename']} | NED: {row['ned']:.2f}", fontsize=10, loc='left')
            except Exception:
                axes[i].axis('off')
                axes[i].text(0.5, 0.5, "Image Missing", ha='center', va='center')

        plt.tight_layout()
        plt.show()